# Empirical Evaluation & Statistical Analysis
**Research Project:** Prototype Evaluation of an AI-Sales-Agent in E-Commerce  
**Author:** Fabian Fürst  
**Date:** June 2026  
**Dataset:** SoSci Survey Export ($N = 68$ valid cases)

---

## 1. Overview & Purpose
This Jupyter Notebook contains the complete quantitative evaluation and statistical analysis for the empirical section of this thesis. The primary goal is to assess the usability, acceptance, and user traits of the developed prototype.

The analysis is structured chronologically into the following steps:
1. **Data Ingestion & Cleaning:** Filtering incomplete responses ($85 \rightarrow 68$) and handling reverse-coded items.
2. **Global Descriptive Statistics:** Calculating mean ($\bar{x}$), median ($\tilde{x}$), and standard deviation ($s$) for all core metrics.
3. **Scale Transformations:** Processing the System Usability Scale (SUS) Composite Score and aggregating Technology Acceptance Model (TAM) dimensions.
4. **Subgroup Analysis:** Conducting a median-split on the technology affinity scale (TAEG) to compare high-affinity vs. low-affinity user groups.
5. **Data Visualization:** Generating publication-ready boxplots and charts exported directly for the thesis.

## 2. System Environment & Prerequisites
This notebook relies on the standard Python data science stack (`pandas`, `numpy`, `matplotlib`, `seaborn`). All outputs are deterministically reproducible.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
from pathlib import Path

csv_path = Path("..") / "data" / "data_usability-akzeptanz-sales-agent.csv"
df = pd.read_csv(csv_path, encoding="utf-16")
df.head()

,CASE,SERIAL,REF,QUESTNNR,MODE,STARTED,DE01_01,DE02,DE03_01,DE04,...,MAILSENT,LASTDATA,STATUS,FINISHED,Q_VIEWER,LASTPAGE,MAXPAGE,MISSING,MISSREL,TIME_RSI
0,Interview-Nummer (fortlaufend),Personenkennung oder Teilnahmecode (sofern ver...,Referenz (sofern im Link angegeben),"Fragebogen, der im Interview verwendet wurde",Interview-Modus,Zeitpunkt zu dem das Interview begonnen hat (E...,Alter: ... Jahre,Geschlecht,Affinität Shopping: [Keine Beschreibung] 01,Endgerät,...,Versandzeitpunkt der Einladungsmail (nur für n...,Zeitpunkt als der Datensatz das letzte mal geä...,Status des Interviews (Markierung),Wurde die Befragung abgeschlossen (letzte Seit...,Hat der Teilnehmer den Fragebogen nur angesehe...,"Seite, die der Teilnehmer zuletzt bearbeitet hat","Letzte Seite, die im Fragebogen bearbeitet wurde",Anteil fehlender Antworten in Prozent,Anteil fehlender Antworten (gewichtet nach Rel...,Ausfüll-Geschwindigkeit (relativ)
1,34,NaN,NaN,base,interview,2026-06-04 20:20:59,26,2,5,1,...,NaN,2026-06-04 20:29:03,complete,1,0,7,7,0,0,1.09
2,36,NaN,NaN,base,interview,2026-06-04 20:35:57,66,1,5,2,...,NaN,2026-06-04 20:43:22,complete,1,0,7,7,0,0,1.04
3,37,NaN,NaN,base,interview,2026-06-04 20:36:00,63,2,5,2,...,NaN,2026-06-04 21:03:43,complete,1,0,7,7,0,0,0.37
4,39,NaN,NaN,base,interview,2026-06-04 20:42:36,23,2,5,2,...,NaN,2026-06-04 20:54:28,complete,1,0,7,7,0,0,0.73


In [3]:
# Drop columns that only contain metadata or are empty after the first row
empty_after_first_row = df.iloc[1:].isna().all(axis=0)
df = df.loc[:, ~empty_after_first_row]
df = df.drop(columns=["CASE", "QUESTNNR", "MODE", "RG01_CP", "LASTPAGE", "MAXPAGE", "MISSING", "MISSREL"])
df.head()

,STARTED,DE01_01,DE02,DE03_01,DE04,EI02,RG01,SU01_01,SU01_02,SU01_03,...,TIME004,TIME005,TIME006,TIME007,TIME_SUM,LASTDATA,STATUS,FINISHED,Q_VIEWER,TIME_RSI
0,Zeitpunkt zu dem das Interview begonnen hat (E...,Alter: ... Jahre,Geschlecht,Affinität Shopping: [Keine Beschreibung] 01,Endgerät,Consent,Zufallsgenerator Szenario: Gezogener Code,"System Usability Scale (SUS): Ich denke, dass ...",System Usability Scale (SUS): Ich fand das Sys...,System Usability Scale (SUS): Ich fand das Sys...,...,Verweildauer Seite 4,Verweildauer Seite 5,Verweildauer Seite 6,Verweildauer Seite 7,Verweildauer gesamt (ohne Ausreißer),Zeitpunkt als der Datensatz das letzte mal geä...,Status des Interviews (Markierung),Wurde die Befragung abgeschlossen (letzte Seit...,Hat der Teilnehmer den Fragebogen nur angesehe...,Ausfüll-Geschwindigkeit (relativ)
1,2026-06-04 20:20:59,26,2,5,1,1,2,5,5,5,...,113,214,49,30,484,2026-06-04 20:29:03,complete,1,0,1.09
2,2026-06-04 20:35:57,66,1,5,2,1,1,4,2,4,...,80,163,71,39,445,2026-06-04 20:43:22,complete,1,0,1.04
3,2026-06-04 20:36:00,63,2,5,2,1,2,5,1,1,...,255,882,261,64,1434,2026-06-04 21:03:43,complete,1,0,0.37
4,2026-06-04 20:42:36,23,2,5,2,1,2,5,1,5,...,140,352,82,39,712,2026-06-04 20:54:28,complete,1,0,0.73


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 46 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   STARTED   86 non-null     str  
 1   DE01_01   77 non-null     str  
 2   DE02      80 non-null     str  
 3   DE03_01   80 non-null     str  
 4   DE04      80 non-null     str  
 5   EI02      86 non-null     str  
 6   RG01      80 non-null     str  
 7   SU01_01   67 non-null     str  
 8   SU01_02   67 non-null     str  
 9   SU01_03   67 non-null     str  
 10  SU01_04   67 non-null     str  
 11  SU01_05   67 non-null     str  
 12  SU01_06   67 non-null     str  
 13  SU01_07   67 non-null     str  
 14  SU01_08   67 non-null     str  
 15  SU01_09   67 non-null     str  
 16  SU01_10   67 non-null     str  
 17  SZ03      69 non-null     str  
 18  SZ03_01   69 non-null     str  
 19  SZ04      71 non-null     str  
 20  SZ04_01   71 non-null     str  
 21  SZ05      69 non-null     str  
 22  SZ06      71 no

In [8]:
# Only use rows where STATUS == 'complete'
df = df[df['STATUS'].fillna('').str.strip().str.lower() == 'complete'].reset_index(drop=True)
df.shape

(66, 46)